# 🎙️ XTTS-v2 — Airi Wake Word Dataset Generator
### Voice Cloning TTS on Google Colab

Generates synthetic `.wav` samples of **"Hello Airi"** using XTTS-v2.

---

## 📋 How to use (read first!)

Run cells **in order, top to bottom**.

1. ▶️ **Cell 1** — installs everything  
2. 🔄 **Restart session** (`Runtime → Restart session`) — mandatory  
3. ▶️ **Cell 2** — configuration  
4. ▶️ **Cell 3** — get reference voices  
5. ▶️ **Cell 4** — load model and encode voices  
6. ▶️ **Cell 5** — generate dataset  
7. ▶️ **Cell 6** — preview a sample  
8. ▶️ **Cell 7** — download files  

> 💡 Set `Runtime → Change runtime type → T4 GPU` before starting.

---

## ⚙️ Cell 1 — Install Everything

**What this does:** Installs all dependencies in one carefully ordered pip call
so that numpy, scipy, librosa, and coqui-tts all share the same ABI-compatible
numpy binary.

**What you do after:** Wait for `ACTION REQUIRED`, then
`Runtime → Restart session`. Do **not** re-run this cell.

> ⚠️ Takes 5–8 minutes. The long output is normal.
>
> **Why one big pip call?**  
> The `numpy dtype size changed` ABI error happens when numpy is reinstalled
> *after* scipy/librosa have already been compiled against a different numpy
> binary in the same session. Installing everything together in one call forces
> pip to resolve a single consistent set of binaries before writing anything.

In [ ]:
import sys, subprocess, os
print(f'Python: {sys.version}')
print()

# ── Single pip call — installs everything together ────────────────────────
#
# WHY ONE CALL:
#   Installing numpy last with --force-reinstall causes an ABI mismatch:
#   scipy and librosa were compiled against the old numpy C headers still
#   resident in memory, so importing numpy after reinstall raises
#   "numpy.dtype size changed, may indicate binary incompatibility".
#   Pip resolves this correctly only when it sees all constraints at once
#   and downloads matching pre-built wheels for the same numpy version.
#
# PIN RATIONALE:
#   numpy<2.0        — coqui-tts internals break on numpy 2.x
#   transformers==4.38.2  — last version without broken 'is_torchcodec_available'
#   tokenizers==0.15.2    — required by transformers 4.38.2; pip otherwise picks 0.21+
#   coqui-tts==0.24.1     — stable model path layout; 0.27.x changed directory names
#   scipy, librosa, soundfile — must be resolved together with numpy so all
#                               get wheels compiled against the same numpy ABI

print('Installing all packages together (5–8 min) ...')
print('(Resolving all versions at once prevents numpy ABI conflicts)')
print()

cmd = [
    sys.executable, '-m', 'pip', 'install', '-q',
    # numpy pin — must be explicit so pip does not pick 2.x
    'numpy>=1.24.0,<2.0.0',
    # transformers + tokenizers pinned together
    'transformers==4.38.2',
    'tokenizers==0.15.2',
    # coqui-tts at a known-good version
    'coqui-tts==0.24.1',
    # audio libs resolved alongside numpy in the same solve
    'scipy',
    'librosa',
    'soundfile',
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print('  pip install: OK')
else:
    print('  pip install: FAILED')
    print(result.stderr[-1500:])

print()

# ── Verification ──────────────────────────────────────────────────────────
# We verify by actually importing the packages — not just checking version
# strings — because the ABI error only surfaces on import.
# NOTE: This verification runs in the CURRENT session which still has old
# binaries loaded. An import error here is expected if Colab had pre-loaded
# packages before Cell 1 ran. The restart clears that state.
# We therefore check versions from the pip metadata, not from importing.

import importlib.metadata as _meta

checks = {
    'numpy'       : ('<2.0',    lambda v: tuple(int(x) for x in v.split('.')[:2]) < (2, 0)),
    'transformers': ('4.38.2',  lambda v: v == '4.38.2'),
    'tokenizers'  : ('0.15.2',  lambda v: v == '0.15.2'),
    'coqui-tts'   : ('0.24.1',  lambda v: v == '0.24.1'),
}

print('=' * 55)
all_ok = True
for pkg, (expected, check_fn) in checks.items():
    try:
        v = _meta.version(pkg)
        ok = check_fn(v)
        mark = '✓' if ok else '✗'
        print(f'  {mark}  {pkg:<16}: {v}  (need {expected})')
        if not ok:
            all_ok = False
    except _meta.PackageNotFoundError:
        print(f'  ✗  {pkg:<16}: NOT INSTALLED')
        all_ok = False

print()
if all_ok:
    print('  ALL INSTALLS COMPLETE!')
    print()
    print('  >>> ACTION REQUIRED <<<')
    print('  Runtime → Restart session')
    print('  Then run Cell 2 onwards — skip Cell 1.')
else:
    print('  One or more packages have wrong versions.')
    print('  Re-run this cell, then restart the session.')
print('=' * 55)


## 🔧 Cell 2 — Configuration

**What you can change:**
- `WAKE_WORD` — phrase to synthesize
- `TARGET_SAMPLES` — how many `.wav` files (start with 10, go to 600 when working)
- `MIN_DURATION` / `MAX_DURATION` — accepted clip length in seconds
- `RANDOM_SEED` — change between runs for variety

**Leave everything else as-is.**

In [ ]:
import os, torch
from pathlib import Path

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ============================================================
#  YOU CAN CHANGE THESE
# ============================================================
WAKE_WORD      = 'Hello Airi'
TARGET_SAMPLES = 10       # start small; set 600 for full dataset
MIN_DURATION   = 1.0      # seconds — clips shorter than this are rejected
MAX_DURATION   = 1.3      # seconds — clips longer than this are rejected
BATCH_SIZE     = 10       # attempts shown per epoch
RANDOM_SEED    = 42       # change each run for variety
# ============================================================

# Fixed — do not change
OUTPUT_DIR  = '/content/xtts_output'
SAMPLES_DIR = '/content/ref_voices'
LANGUAGE    = 'en'
OUTPUT_SR   = 24000   # XTTS-v2 native output sample rate

# coqui-tts 0.24.1 stores the model here after first download
MODEL_DIR   = '/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2'

# ── XTTS-v2 native variation knobs ───────────────────────────────────────
# All of these are passed directly to model.inference() — no post-processing.
# temperature : GPT randomness (0.6 = stable/predictable, 0.85 = expressive)
# top_k       : top-K candidates per GPT step
# top_p       : nucleus sampling cutoff
# rep_penalty : penalises repeated tokens; prevents silence loops
# speed       : maps to 1/length_scale inside HiFiGAN — native speed control
TEMPERATURE_RANGE = [0.60, 0.70, 0.75, 0.80, 0.85]
TOP_K_RANGE       = [30, 40, 50, 60, 70]
TOP_P_RANGE       = [0.75, 0.80, 0.85, 0.90, 0.95]
REP_PENALTY_RANGE = [3.0, 5.0, 7.0, 9.0]
SPEED_RANGE       = [0.85, 0.90, 1.00, 1.05, 1.10]

if torch.cuda.is_available():
    DEVICE  = 'cuda'
    USE_GPU = True
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print('Generation will be fast (~3–8 s/sample).')
else:
    DEVICE  = 'cpu'
    USE_GPU = False
    print('No GPU — CPU mode (~60–120 s/sample).')
    print('Tip: Runtime → Change runtime type → T4 GPU')

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(SAMPLES_DIR).mkdir(parents=True, exist_ok=True)

print()
print('Configuration set!')
print(f'  Wake word      : {WAKE_WORD!r}')
print(f'  Target samples : {TARGET_SAMPLES}')
print(f'  Duration range : {MIN_DURATION}s – {MAX_DURATION}s')
print(f'  Output folder  : {OUTPUT_DIR}')


## 🎤 Cell 3 — Get Reference Voices

**Option A (recommended):** Auto-downloads 5 diverse voices from the NeuTTS repo.  
**Option B:** Upload your own `.wav` files (6–30 s of clear speech each).

Run only **one** of the two blocks below.

In [ ]:
# ── OPTION A: Download built-in voices (recommended) ─────────────────────
import shutil, subprocess
from pathlib import Path

REPO_DIR = '/tmp/neutts_repo'
shutil.rmtree(REPO_DIR, ignore_errors=True)

print('Downloading sample voices (voice files only, not the whole repo) ...')
r = subprocess.run(
    ['git', 'clone', '--depth=1', '--filter=blob:none', '--sparse',
     'https://github.com/neuphonic/neutts-air.git', REPO_DIR],
    capture_output=True, text=True
)
if r.returncode != 0:
    print('Clone failed:', r.stderr)
else:
    subprocess.run(
        ['git', '-C', REPO_DIR, 'sparse-checkout', 'set', 'samples'],
        capture_output=True
    )
    copied = 0
    for f in (Path(REPO_DIR) / 'samples').glob('*'):
        shutil.copy(f, Path(SAMPLES_DIR) / f.name)
        if f.suffix == '.wav':
            print(f'  Copied: {f.name}')
            copied += 1
    print(f'\n{copied} voice file(s) ready.')

ref_wavs = list(Path(SAMPLES_DIR).glob('*.wav'))
print(f'Total reference voices: {len(ref_wavs)}')
for w in ref_wavs:
    print(f'  - {w.name}')


In [ ]:
# ── OPTION B: Upload your own voices (skip if you ran Option A) ───────────
from google.colab import files
from pathlib import Path

print('Upload .wav files (6–30 s each, clear speech).')
uploaded = files.upload()
for fname, data in uploaded.items():
    (Path(SAMPLES_DIR) / fname).write_bytes(data)
    print(f'  Saved: {fname}')
print(f'Total reference voices: {len(list(Path(SAMPLES_DIR).glob("*.wav")))}')


## 📦 Cell 4 — Load XTTS-v2 and Encode Voices

**What this does:**
1. Downloads XTTS-v2 weights (~2.1 GB, first run only — cached after)
2. Applies a one-line patch inside `TTS/utils/io.py` to fix a PyTorch 2.6 incompatibility
3. Encodes each reference voice into speaker embeddings (done once, reused for all samples)

**The patch:** PyTorch 2.6 changed `torch.load` to default `weights_only=True`.
XTTS-v2 checkpoints contain Python class instances (not just tensors), so they raise
`UnpicklingError` under the new default. The patch forces `weights_only=False` in
the one internal function that loads the checkpoint. This is safe — the checkpoint
comes from the official Coqui / HuggingFace release.

**What you do:** Run it and wait. First run downloads ~2.1 GB.

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from pathlib import Path
import torch

# ── 0. Safety check ───────────────────────────────────────────────────────
ref_wavs = sorted(Path(SAMPLES_DIR).glob('*.wav'))
if not ref_wavs:
    raise FileNotFoundError(
        f'No .wav files in {SAMPLES_DIR}. Run Cell 3 first.'
    )

# ── 1. Import TTS internals ───────────────────────────────────────────────
try:
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.tts.models.xtts import Xtts
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        'coqui-tts not found. Run Cell 1, restart session, then try again.'
    )

# ── 2. Patch torch.load inside TTS to force weights_only=False ───────────
#
# ROOT CAUSE (confirmed from error traceback):
#   File TTS/utils/io.py line 54:
#     return torch.load(f, map_location=map_location, **kwargs)
#   PyTorch >= 2.6 defaults weights_only=True.
#   XTTS checkpoint contains TTS.config.shared_configs.BaseDatasetConfig
#   (a Python object, not a tensor). This raises UnpicklingError.
#
# FIX: Monkey-patch load_fsspec to always pass weights_only=False.
#   We patch both the module-level reference and the already-imported
#   reference inside the xtts module to handle Python import caching.

import fsspec
import TTS.utils.io  as _tts_io
import TTS.tts.models.xtts as _xtts_mod

def _patched_load_fsspec(path, map_location=None, cache=True, **kwargs):
    """Patched load_fsspec: forces weights_only=False for PyTorch >= 2.6."""
    kwargs['weights_only'] = False   # override whatever caller passed
    with fsspec.open(path, 'rb') as f:
        return torch.load(f, map_location=map_location, **kwargs)

# Patch both locations to be safe against import caching
_tts_io.load_fsspec   = _patched_load_fsspec
_xtts_mod.load_fsspec = _patched_load_fsspec

print('Patch applied: weights_only=False for XTTS checkpoint loading.')
print()

# ── 3. Download model if not cached ───────────────────────────────────────
config_path = Path(MODEL_DIR) / 'config.json'

if not config_path.exists():
    print('Model not in cache — downloading (~2.1 GB, first run only) ...')
    print('This takes 3–5 minutes. Please wait.')
    # Set the env var that coqui-tts 0.24.1 checks before showing the
    # interactive license prompt. Without this the prompt blocks execution.
    os.environ['COQUI_TOS_AGREED'] = '1'
    from TTS.api import TTS as _Downloader
    # gpu=False here — we load manually below with the patched loader
    _Downloader('tts_models/multilingual/multi-dataset/xtts_v2', gpu=False)
    del _Downloader
    print('Download complete.')
    print()

# Fallback: search the TTS cache directory in case the path differs slightly
if not config_path.exists():
    tts_cache = Path('/root/.local/share/tts')
    candidates = [p for p in tts_cache.rglob('config.json')
                  if 'xtts_v2' in str(p)]
    if candidates:
        config_path = candidates[0]
        model_dir   = config_path.parent
        print(f'Found model at: {model_dir}')
    else:
        raise RuntimeError(
            'config.json not found after download attempt.\n'
            'The download may have been interrupted. Try:\n'
            '  !rm -rf /root/.local/share/tts\n'
            'Then restart the session and run from Cell 2.'
        )
else:
    model_dir = Path(MODEL_DIR)

print(f'Model directory : {model_dir}')
print(f'Config          : {config_path}')
print()

# ── 4. Load model ─────────────────────────────────────────────────────────
print('Loading model into memory ...')

config = XttsConfig()
config.load_json(str(config_path))

model = Xtts.init_from_config(config)
model.load_checkpoint(
    config,
    checkpoint_dir = str(model_dir),
    eval           = True,
    use_deepspeed  = False,
)

if USE_GPU:
    model.cuda()

print(f'Model loaded!  Device: {DEVICE.upper()}')
print()

# ── 5. Encode reference voices (once — reused for all samples) ────────────
print(f'Encoding {len(ref_wavs)} reference voice(s) ...')
ENCODED_REFS = []

for wav in ref_wavs:
    try:
        gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
            audio_path         = [str(wav)],
            gpt_cond_len       = 12,  # seconds used for GPT conditioning
            gpt_cond_chunk_len = 4,   # chunk size for latent averaging
            max_ref_length     = 30,
            sound_norm_refs    = False,
            librosa_trim_db    = 20,  # trim silence from reference before encoding
            load_sr            = 24000,
        )
        ENCODED_REFS.append({
            'name'             : wav.stem,
            'gpt_cond_latent'  : gpt_cond_latent,
            'speaker_embedding': speaker_embedding,
        })
        print(f'  Encoded: {wav.name}')
    except Exception as e:
        print(f'  SKIP {wav.name} — {e}')

if not ENCODED_REFS:
    raise RuntimeError(
        'No voice could be encoded. '
        'Check that .wav files are valid and at least 6 s long.'
    )

print()
print(f'All {len(ENCODED_REFS)} voice(s) encoded and ready.')
print('Run Cell 5 to generate your dataset!')


## 🎙️ Cell 5 — Generate Your Dataset

For each sample the loop:
1. Picks a random voice and random XTTS-native parameters
2. Synthesizes with XTTS-v2's GPT + HiFiGAN pipeline
3. Trims silence from output edges (prevents padding from inflating duration)
4. Rejects clips outside the duration window
5. Saves accepted clips with parameter-encoded filenames

`OK` = saved. `SKIP` = wrong duration (normal — expect 20–40% rejection).

> ⏱️ T4 GPU: ~3–8 s/attempt → 600 samples in ~30–60 min.

In [ ]:
import numpy as np
import random
import time
import soundfile as sf
import librosa
from pathlib import Path

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

output_path      = Path(OUTPUT_DIR)
valid_samples    = 0
rejected_samples = 0
total_attempts   = 0
epoch            = 1
start_time       = time.time()

print('=' * 60)
print('  AIRI WAKE WORD GENERATION — XTTS-v2')
print('=' * 60)
print(f'  Wake word       : {WAKE_WORD!r}')
print(f'  Target samples  : {TARGET_SAMPLES}')
print(f'  Duration range  : {MIN_DURATION}s – {MAX_DURATION}s')
print(f'  Reference voices: {len(ENCODED_REFS)}')
print(f'  Device          : {DEVICE.upper()}')
print('=' * 60)


def trim_silence(audio: np.ndarray, sr: int, top_db: float = 30.0) -> np.ndarray:
    """Trim leading/trailing silence to prevent padding from inflating duration."""
    trimmed, _ = librosa.effects.trim(audio, top_db=top_db)
    return trimmed


while valid_samples < TARGET_SAMPLES:
    remaining         = TARGET_SAMPLES - valid_samples
    target_this_batch = min(BATCH_SIZE, remaining)
    print(f'\n--- Epoch {epoch} | {valid_samples}/{TARGET_SAMPLES} done | '
          f'{remaining} to go ---')

    bv = br = 0
    attempts_this_epoch = 0

    while bv < target_this_batch and attempts_this_epoch < BATCH_SIZE * 4:
        ref         = random.choice(ENCODED_REFS)
        temperature = random.choice(TEMPERATURE_RANGE)
        top_k       = random.choice(TOP_K_RANGE)
        top_p       = random.choice(TOP_P_RANGE)
        rep_penalty = random.choice(REP_PENALTY_RANGE)
        speed       = random.choice(SPEED_RANGE)

        total_attempts      += 1
        attempts_this_epoch += 1

        try:
            # model.inference() passes all parameters through to the GPT
            # backbone — unlike the TTS() facade which silently drops speed.
            out = model.inference(
                text                  = WAKE_WORD,
                language              = LANGUAGE,
                gpt_cond_latent       = ref['gpt_cond_latent'],
                speaker_embedding     = ref['speaker_embedding'],
                temperature           = temperature,
                length_penalty        = 1.0,
                repetition_penalty    = rep_penalty,
                top_k                 = top_k,
                top_p                 = top_p,
                do_sample             = True,
                speed                 = speed,
                enable_text_splitting = False,
            )

            # out['wav'] is a Python list of float32 values at OUTPUT_SR
            audio = np.array(out['wav'], dtype=np.float32)

            # Trim silence BEFORE measuring duration.
            # XTTS often appends a short silence tail; without trimming, many
            # valid clips would fall just outside the duration window.
            audio = trim_silence(audio, OUTPUT_SR, top_db=30.0)

            dur = len(audio) / OUTPUT_SR

            if not (MIN_DURATION <= dur <= MAX_DURATION):
                br += 1
                print(f'  SKIP [{attempts_this_epoch:>3}] {dur:.2f}s — '
                      f'outside {MIN_DURATION}–{MAX_DURATION}s')
            else:
                ts    = int(time.time() * 1000)
                fname = (
                    f"airi_{valid_samples:04d}_{ts}_"
                    f"{ref['name'][:8]}_"
                    f"t{temperature:.2f}_k{top_k}_"
                    f"p{top_p:.2f}_r{rep_penalty:.0f}_"
                    f"spd{speed:.2f}.wav"
                )
                sf.write(str(output_path / fname), audio, OUTPUT_SR)
                bv            += 1
                valid_samples += 1
                print(f"  OK  [{attempts_this_epoch:>3}] "
                      f"voice={ref['name']:<10} "
                      f"t={temperature:.2f} k={top_k} "
                      f"p={top_p:.2f} r={rep_penalty:.0f} "
                      f"spd={speed:.2f} dur={dur:.2f}s")

                if valid_samples >= TARGET_SAMPLES:
                    break

        except Exception as e:
            br += 1
            print(f'  ERR [{attempts_this_epoch:>3}] {e}')

    rejected_samples += br
    epoch            += 1

    elapsed = time.time() - start_time
    rate    = valid_samples / elapsed if elapsed > 0 else 0
    eta     = (TARGET_SAMPLES - valid_samples) / rate if rate > 0 else float('inf')
    print(f'  Batch: {bv} saved, {br} skipped | '
          f'Total: {valid_samples}/{TARGET_SAMPLES} | '
          f'Elapsed: {elapsed/60:.1f}min | ETA: {eta/60:.1f}min')


elapsed = time.time() - start_time
print('\n' + '=' * 60)
print('  GENERATION COMPLETE!')
print('=' * 60)
print(f'  Valid samples   : {valid_samples}')
print(f'  Rejected/skipped: {rejected_samples}')
print(f'  Success rate    : {valid_samples / max(total_attempts, 1) * 100:.1f}%')
print(f'  Total time      : {elapsed / 60:.1f} minutes')
print(f'  Avg per sample  : {elapsed / max(valid_samples, 1):.2f}s')
print(f'  Files saved to  : {OUTPUT_DIR}')
print('=' * 60)
print()
print('Run Cell 6 to preview, or Cell 7 to download.')


## 🔍 Cell 6 — Preview a Sample (Optional)

Plays a random clip from your generated dataset. Quick sanity check before downloading.

In [ ]:
from IPython.display import Audio, display
import random as _rnd
from pathlib import Path

wav_files = list(Path(OUTPUT_DIR).glob('*.wav'))
if not wav_files:
    print('No .wav files yet — run Cell 5 first.')
else:
    sample = _rnd.choice(wav_files)
    print(f'Playing: {sample.name}')
    print(f'Total files: {len(wav_files)}')
    display(Audio(str(sample)))


## 📥 Cell 7 — Download Your Dataset

⚠️ Colab deletes `/content/` when the session ends — download before closing!

**Option A** — zip directly to your computer.  
**Option B** — save to Google Drive (safer for large datasets).

In [ ]:
# ── OPTION A: Download zip ────────────────────────────────────────────────
import zipfile
from google.colab import files
from pathlib import Path

zip_path  = '/content/airi_xtts_dataset.zip'
wav_files = list(Path(OUTPUT_DIR).glob('*.wav'))

if not wav_files:
    print('No .wav files — run Cell 5 first.')
else:
    print(f'Zipping {len(wav_files)} files ...')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in wav_files:
            zf.write(f, f.name)
    size_mb = Path(zip_path).stat().st_size / (1024 ** 2)
    print(f'Zip ready: {size_mb:.1f} MB — starting download ...')
    files.download(zip_path)


In [ ]:
# ── OPTION B: Save to Google Drive ───────────────────────────────────────
import shutil, os
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_DEST = '/content/drive/MyDrive/airi_xtts_dataset'
os.makedirs(DRIVE_DEST, exist_ok=True)

wav_files = list(Path(OUTPUT_DIR).glob('*.wav'))
if not wav_files:
    print('No .wav files — run Cell 5 first.')
else:
    print(f'Copying {len(wav_files)} files to Google Drive ...')
    for f in wav_files:
        shutil.copy(f, DRIVE_DEST)
    print(f'Done! Files saved to: My Drive/airi_xtts_dataset/')
